# ChaoSSM — Phase 0c: Binding the Constraint & Testing the Edge-of-Chaos Claim

## What 0b settled

- **§4.3 Option A is disqualified analytically.** Ceiling `log σ_max = −0.029`; at typical states
  −1.07. No hyperparameter recovers it. The outer sigmoid caps `σ' ≤ 0.25` and `r ≤ 4`, so the
  product cannot exceed 1.
- **`fold` works as the replacement.** Triangle-fold bounds the state without contracting it:
  λ̂ went positive (+0.59 to +0.74) and the clamp-activation rate collapsed from ~68% to ~3%.
- **`gain` fails in practice** despite passing the Stage A screen — during training the state
  distribution is shaped by input injection and gating, and the sigmoid still squashes where
  trajectories actually live. Stage A is necessary, not sufficient.
- **Fixed α cannot bind the constraint.** With `fold`, all three arms land at λ̂ ≈ 0.62–0.66 and
  spend **0%** of training inside the corridor. The regularizer is ~1% of the loss. This is not
  "α is a bit low" — the architecture sets the unregularized operating point, and a fixed
  coefficient is the wrong instrument for holding a constraint.

## What 0c does differently

**1. Dual ascent instead of a fixed α.** Treat the corridor as a constraint and let the
multiplier find its own scale:

```
L = L_task + α_hi·max(0, λ̂ − hi) + α_lo·max(0, lo − λ̂)
α_hi ← max(0, α_hi + η·(λ̂ − hi))        # grows until the constraint binds
α_lo ← max(0, α_lo + η·(lo − λ̂))
```

This removes the guesswork. If the corridor is reachable, α grows until λ̂ is pulled in; if it
is *not* reachable, α diverges and that is itself the finding.

**2. A λ-target sweep instead of three arms.** The corridor-vs-floss-vs-free comparison assumed
the corridor could be occupied. It can't be, so that comparison measured nothing. The scientific
question underneath it is better asked directly:

> **Does task performance peak near λ ≈ 0?**

So we enforce a range of λ targets and measure accuracy at each. If performance peaks near the
edge of chaos, §3.2's expressivity argument is supported. If it is flat, or peaks at the
unregularized operating point, it is not — and that is a publishable result either way.

**3. A learnability gate.** Every 0b run finished at ~0.057 accuracy, which is exactly chance for
18 classes. A λ-vs-performance sweep is meaningless if the task is never learned, so 0c checks
that the unconstrained model gets above chance *before* running the sweep.

## Setup

In [11]:
import os, json, math, time, warnings
from dataclasses import dataclass, asdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import wilcoxon
from torch.func import jacrev, vmap

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE)

def setup_results_dir(project="ChaoSSM", phase="phase0c"):
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        base = f"/content/drive/My Drive/{project}/results/{phase}"
    except Exception as e:
        base = os.path.abspath(f"./{project}_results/{phase}")
        print(f"[info] Not in Colab ({type(e).__name__}); using local dir.")
    os.makedirs(base, exist_ok=True); print("results ->", base); return base

RESULTS = setup_results_dir()
def savefig(f,n): p=os.path.join(RESULTS,n); f.savefig(p,dpi=300,bbox_inches="tight"); print("saved",p)
def savetable(d,n): p=os.path.join(RESULTS,n); d.to_csv(p,index=False); print("saved",p)
def savejson(o,n):
    p=os.path.join(RESULTS,n)
    with open(p,"w") as fh: json.dump(o,fh,indent=2,default=str)
    print("saved",p)

torch 2.11.0+cu128 | device: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
results -> /content/drive/My Drive/ChaoSSM/results/phase0c


## Configuration

`copy_len` is reduced to 8 and `steps` raised to 2000 so the task is actually learnable — 0b's
16-token copy at 60 steps never left chance.

`lyap_window=4` and `lyap_subsample=4` cut the estimator from 48 Jacobian evaluations per step
to 16. Both still beat the biased single-step estimator; raise them if runtime allows.

In [12]:
SMOKE_TEST = False   # full run: learnability gate -> 5-target lambda sweep

@dataclass
class Config:
    state_dim: int = 48
    d_model: int = 48
    vocab: int = 16
    copy_len: int = 8            # was 16 — 0b never left chance
    steps: int = 2000            # was 600
    batch: int = 64
    lr: float = 3e-4
    bptt_k: int = 64
    grad_clip: float = 1.0
    variant: str = "fold"        # settled in 0b
    # --- dual ascent
    dual_lr: float = 0.05        # eta
    alpha_max: float = 50.0      # divergence guard
    corridor_half_width: float = 0.10
    # --- estimator
    estimator: str = "window"
    lyap_window: int = 4
    lyap_subsample: int = 4
    lyap_every: int = 10         # logging cadence only
    benettin_transient: int = 100
    benettin_measure: int = 300
    # --- sweep
    lam_targets: tuple = (-0.20, 0.00, 0.15, 0.40, 0.65)
    seeds: int = 5
    free_seeds: int = 5          # unconstrained reference arm

CFG = Config()
if SMOKE_TEST:
    CFG.steps = 80; CFG.seeds = 1; CFG.free_seeds = 1
    CFG.lam_targets = (0.00, 0.65)
    CFG.benettin_measure = 60; CFG.benettin_transient = 20
    print("*** SMOKE TEST — pipeline check only ***")
print(json.dumps({k: str(v) for k, v in asdict(CFG).items()}, indent=2))

{
  "state_dim": "48",
  "d_model": "48",
  "vocab": "16",
  "copy_len": "8",
  "steps": "2000",
  "batch": "64",
  "lr": "0.0003",
  "bptt_k": "64",
  "grad_clip": "1.0",
  "variant": "fold",
  "dual_lr": "0.05",
  "alpha_max": "50.0",
  "corridor_half_width": "0.1",
  "estimator": "window",
  "lyap_window": "4",
  "lyap_subsample": "4",
  "lyap_every": "10",
  "benettin_transient": "100",
  "benettin_measure": "300",
  "lam_targets": "(-0.2, 0.0, 0.15, 0.4, 0.65)",
  "seeds": "5",
  "free_seeds": "5"
}


## 1. Model (fold variant, carried over from 0b)

In [13]:
def triangle_fold(x):
    '''R -> [0,1], piecewise linear, |derivative| == 1 everywhere.'''
    return 1.0 - torch.abs(torch.remainder(x, 2.0) - 1.0)


class ChaoticMap(nn.Module):
    def __init__(self, N):
        super().__init__()
        self.w_r = nn.Parameter(torch.zeros(N))
        W = torch.empty(N, N); nn.init.orthogonal_(W)
        self.W_mix = nn.Parameter(W * 0.1)
    def get_r(self):
        return 3.57 + 0.43 * torch.sigmoid(self.w_r)
    def forward(self, h):
        r = self.get_r()
        return triangle_fold(h @ self.W_mix.T + r * h * (1 - h))


class ChaoSSMCell(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.N = cfg.state_dim
        self.phi = ChaoticMap(cfg.state_dim)
        self.A = nn.Parameter(torch.diag(torch.empty(cfg.state_dim).uniform_(0.9, 1.1)))
        self.B = nn.Parameter(torch.randn(cfg.state_dim, cfg.d_model) / math.sqrt(cfg.d_model))
        self.W_g = nn.Parameter(torch.randn(cfg.state_dim, cfg.d_model) / math.sqrt(cfg.d_model))
        self.b_g = nn.Parameter(torch.zeros(cfg.state_dim))
    def step(self, h, x):
        g = torch.sigmoid(x @ self.W_g.T + self.b_g)
        return triangle_fold(g * self.phi(h @ self.A.T) + (1 - g) * h + x @ self.B.T)
    def init_state(self, b, dev):
        return torch.full((b, self.N), 0.5, device=dev)


class ChaoSSMLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.emb = nn.Embedding(cfg.vocab + 2, cfg.d_model)
        self.cell = ChaoSSMCell(cfg)
        self.readout = nn.Linear(cfg.state_dim, cfg.vocab + 2)
    def forward(self, tok, return_traj=False):
        B, T = tok.shape
        x = self.emb(tok); h = self.cell.init_state(B, tok.device)
        outs, traj = [], []
        for t in range(T):
            h = self.cell.step(h, x[:, t]); outs.append(self.readout(h))
            if return_traj: traj.append(h)
            if (t + 1) % self.cfg.bptt_k == 0: h = h.detach()
        lg = torch.stack(outs, 1)
        return (lg, torch.stack(traj, 1), x) if return_traj else lg


def make_copy_batch(cfg, batch, device, gen):
    L, V = cfg.copy_len, cfg.vocab
    src = torch.randint(0, V, (batch, L), device=device, generator=gen)
    inp = torch.cat([src, torch.full((batch,1), V, device=device),
                     torch.full((batch,L), V+1, device=device)], 1)
    tgt = torch.cat([torch.full((batch,L+1), -100, device=device), src], 1)
    return inp, tgt

CHANCE = 1.0 / (CFG.vocab + 2)
print(f"chance accuracy = {CHANCE:.4f}")

chance accuracy = 0.0556


## 2. Estimators

In [14]:
def _full_jac_fn(model, x):
    return lambda z: model.cell.step(z.unsqueeze(0), x.unsqueeze(0))[0]


def lambda_window(model, traj, xseq, W, n_sample):
    '''Finite-time Lyapunov exponent of the FULL recurrence over W steps.'''
    B, T, N = traj.shape
    lams = []
    for b in torch.randperm(B)[:n_sample]:
        t0 = int(torch.randint(0, max(1, T - W - 1), (1,)))
        M = torch.eye(N, device=traj.device, dtype=traj.dtype)
        for k in range(W):
            M = jacrev(_full_jac_fn(model, xseq[b, t0+k]))(traj[b, t0+k]) @ M
        lams.append(torch.log(torch.linalg.matrix_norm(M, ord=2) + 1e-12) / W)
    return torch.stack(lams).mean()


@torch.no_grad()
def benettin_spectrum(step_fn, h0, inputs, N, T_tr, T_me):
    h = h0.clone()
    for t in range(T_tr): h = step_fn(h, inputs[t % len(inputs)])
    Q = torch.eye(N, device=h.device, dtype=h.dtype); lyap = torch.zeros(N, device=h.device, dtype=h.dtype)
    for t in range(T_me):
        x = inputs[(T_tr + t) % len(inputs)]
        Q = jacrev(lambda hh: step_fn(hh, x))(h) @ Q
        Q, R = torch.linalg.qr(Q)
        lyap += torch.log(torch.abs(torch.diagonal(R)) + 1e-12)
        h = step_fn(h, x)
    return (lyap / T_me).sort(descending=True).values


def kaplan_yorke(sp):
    s = sp.detach().cpu().numpy().astype(np.float64); c = np.cumsum(s)
    idx = np.where(c >= 0)[0]
    if len(idx) == 0: return 0.0
    j = int(idx[-1]) + 1
    if j >= len(s): return float(len(s))
    d = abs(s[j])
    return float(j) if d < 1e-12 else float(j + c[j-1]/d)


# --- validation on a system with a known answer (lambda = ln 2)
f = lambda h: 4.0 * h * (1 - h)
sp = benettin_spectrum(lambda hh, x: f(hh), torch.tensor([0.31234], dtype=torch.float64),
                       [torch.zeros(1, dtype=torch.float64)], 1, 500, 2000)[0].item()
print(f"Benettin on logistic r=4: {sp:.5f}  (truth {math.log(2):.5f})  "
      f"{'PASS' if abs(sp-math.log(2))<0.02 else 'FAIL'}")

Benettin on logistic r=4: 0.69259  (truth 0.69315)  PASS


## 3. Trainer with dual ascent

`lam_target=None` runs unconstrained. Otherwise the corridor is
`[target − half_width, target + half_width]`, with multipliers grown by dual ascent until the
constraint binds. `alpha_hi`/`alpha_lo` are logged so a diverging multiplier — i.e. a target the
architecture cannot reach — is visible rather than silent.

In [15]:
def train_one(lam_target, seed, cfg):
    torch.manual_seed(seed); np.random.seed(seed)
    gen = torch.Generator(device=DEVICE); gen.manual_seed(seed)
    model = ChaoSSMLM(cfg).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, betas=(0.9, 0.95))

    lo = hi = None
    if lam_target is not None:
        lo, hi = lam_target - cfg.corridor_half_width, lam_target + cfg.corridor_half_width
    a_hi = a_lo = 0.0
    hist, diverged = [], False

    for step in range(cfg.steps):
        inp, tgt = make_copy_batch(cfg, cfg.batch, DEVICE, gen)
        logits, traj, xseq = model(inp, return_traj=True)
        L_task = F.cross_entropy(logits.reshape(-1, logits.shape[-1]),
                                 tgt.reshape(-1), ignore_index=-100)
        lam = lambda_window(model, traj, xseq, cfg.lyap_window, cfg.lyap_subsample)

        if lam_target is None:
            L_pen = torch.zeros((), device=DEVICE)
        else:
            L_pen = a_hi * F.relu(lam - hi) + a_lo * F.relu(lo - lam)

        opt.zero_grad(set_to_none=True); (L_task + L_pen).backward()
        gn = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip).item()
        opt.step()

        # --- dual ascent on the multipliers
        if lam_target is not None:
            lv = float(lam.detach())
            a_hi = float(np.clip(a_hi + cfg.dual_lr * (lv - hi), 0.0, cfg.alpha_max))
            a_lo = float(np.clip(a_lo + cfg.dual_lr * (lo - lv), 0.0, cfg.alpha_max))

        if not (math.isfinite(L_task.item()) and math.isfinite(gn)):
            diverged = True; break

        if step % cfg.lyap_every == 0:
            with torch.no_grad():
                acc = (logits[:, cfg.copy_len+1:].argmax(-1)
                       == tgt[:, cfg.copy_len+1:]).float().mean().item()
            hist.append(dict(step=step, lam_target=lam_target if lam_target is not None else np.nan,
                             seed=seed, loss=L_task.item(), acc=acc, lam=float(lam),
                             gnorm=gn, alpha_hi=a_hi, alpha_lo=a_lo,
                             in_corridor=bool(lam_target is not None and lo <= float(lam) <= hi)))

    # --- post-hoc reference spectrum
    model.eval()
    with torch.no_grad():
        inp, _ = make_copy_batch(cfg, 1, DEVICE, gen)
        xs = [model.emb(inp[0, t]).double() for t in range(inp.shape[1])]
    cf = model.cell.double(); h0 = cf.init_state(1, DEVICE)[0].double()
    spec = benettin_spectrum(lambda hh, x: cf.step(hh.unsqueeze(0), x.unsqueeze(0))[0],
                             h0, xs, cfg.state_dim, cfg.benettin_transient, cfg.benettin_measure)

    tail = hist[-max(1, len(hist)//5):]     # last 20% = settled regime
    s = dict(lam_target=lam_target, seed=seed, diverged=diverged,
             final_acc=float(np.mean([h["acc"] for h in tail])) if tail else np.nan,
             best_acc=float(np.max([h["acc"] for h in hist])) if hist else np.nan,
             lam_settled=float(np.mean([h["lam"] for h in tail])) if tail else np.nan,
             frac_in_corridor=float(np.mean([h["in_corridor"] for h in tail])) if tail else np.nan,
             alpha_hi_final=hist[-1]["alpha_hi"] if hist else np.nan,
             alpha_lo_final=hist[-1]["alpha_lo"] if hist else np.nan,
             alpha_saturated=bool(hist and max(hist[-1]["alpha_hi"], hist[-1]["alpha_lo"])
                                  >= cfg.alpha_max * 0.99),
             lam1_benettin=float(spec[0]), DKY=kaplan_yorke(spec))
    return s, pd.DataFrame(hist)

## 4. Learnability gate

Train the unconstrained model and confirm it beats chance. If it does not, the λ-sweep cannot
be interpreted and the task or budget must change before proceeding.

In [16]:
free_rows, free_hist = [], []
for sd in range(CFG.free_seeds):
    s, h = train_one(None, sd, CFG)
    free_rows.append(s); free_hist.append(h)
    print(f"  free seed={sd}  acc={s['final_acc']:.4f} (chance {CHANCE:.4f})  "
          f"best={s['best_acc']:.4f}  lam={s['lam_settled']:+.4f}  DKY={s['DKY']:.2f}")

FREE = pd.DataFrame(free_rows); FREEH = pd.concat(free_hist, ignore_index=True)
savetable(FREE, "phase0c_free_reference.csv")

learned = bool(FREE.final_acc.mean() > CHANCE * 1.5)
print(f"\nLEARNABILITY GATE: {'PASS' if learned else 'FAIL'} "
      f"(mean acc {FREE.final_acc.mean():.4f} vs chance {CHANCE:.4f})")
if not learned:
    print("!! The unconstrained model has not learned the task. The lambda-vs-performance\n"
          "   sweep below is UNINTERPRETABLE until this passes. Increase steps, shorten\n"
          "   copy_len, or raise lr before drawing any conclusion from Section 5.")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
d = FREEH.groupby("step")[["acc", "lam"]].mean()
ax[0].plot(d.index, d.acc); ax[0].axhline(CHANCE, color="r", ls="--", label="chance")
ax[0].set(xlabel="step", ylabel="copy accuracy", title="Unconstrained learnability"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(d.index, d.lam); ax[1].axhline(0, color="k", ls=":")
ax[1].set(xlabel="step", ylabel=r"$\hat\lambda$", title="Unconstrained operating point"); ax[1].grid(alpha=.3)
fig.tight_layout(); savefig(fig, "phase0c_learnability.png"); plt.close(fig)

  free seed=0  acc=0.0712 (chance 0.0556)  best=0.0996  lam=+1.0294  DKY=33.85
  free seed=1  acc=0.0667 (chance 0.0556)  best=0.1133  lam=+0.9153  DKY=29.25
  free seed=2  acc=0.0727 (chance 0.0556)  best=0.1270  lam=+1.0798  DKY=26.34
  free seed=3  acc=0.0692 (chance 0.0556)  best=0.0996  lam=+1.0420  DKY=37.83
  free seed=4  acc=0.0704 (chance 0.0556)  best=0.1055  lam=+1.0902  DKY=36.55
saved /content/drive/My Drive/ChaoSSM/results/phase0c/phase0c_free_reference.csv

LEARNABILITY GATE: FAIL (mean acc 0.0700 vs chance 0.0556)
!! The unconstrained model has not learned the task. The lambda-vs-performance
   sweep below is UNINTERPRETABLE until this passes. Increase steps, shorten
   copy_len, or raise lr before drawing any conclusion from Section 5.
saved /content/drive/My Drive/ChaoSSM/results/phase0c/phase0c_learnability.png


## 5. λ-target sweep

Two questions at once: **can** dual ascent hold each target (`frac_in_corridor`, and whether the
multiplier saturates), and **should** it — does accuracy peak near λ ≈ 0?

In [17]:
rows, hists = [], []
for tgt in CFG.lam_targets:
    for sd in range(CFG.seeds):
        s, h = train_one(tgt, sd, CFG)
        rows.append(s); hists.append(h)
        flag = " ALPHA-SATURATED" if s["alpha_saturated"] else ""
        print(f"  target={tgt:+.2f} seed={sd}  lam={s['lam_settled']:+.4f}  "
              f"in_corridor={s['frac_in_corridor']:.0%}  acc={s['final_acc']:.4f}  "
              f"DKY={s['DKY']:.2f}  a_hi={s['alpha_hi_final']:.2f}{flag}")

SW = pd.DataFrame(rows); SWH = pd.concat(hists, ignore_index=True)
savetable(SW, "phase0c_sweep.csv"); savetable(SWH, "phase0c_sweep_history.csv")
print()
print(SW.groupby("lam_target")[["lam_settled","frac_in_corridor","final_acc","DKY",
                                "alpha_hi_final","alpha_lo_final"]].mean().round(4).to_string())

  target=-0.20 seed=0  lam=+1.1418  in_corridor=0%  acc=0.0658  DKY=36.33  a_hi=50.00 ALPHA-SATURATED
  target=-0.20 seed=1  lam=+0.5906  in_corridor=0%  acc=0.0604  DKY=7.98  a_hi=50.00 ALPHA-SATURATED
  target=-0.20 seed=2  lam=+0.3994  in_corridor=0%  acc=0.0661  DKY=1.44  a_hi=50.00 ALPHA-SATURATED
  target=-0.20 seed=3  lam=+1.1084  in_corridor=0%  acc=0.0646  DKY=33.67  a_hi=50.00 ALPHA-SATURATED
  target=-0.20 seed=4  lam=+0.5195  in_corridor=0%  acc=0.0625  DKY=2.89  a_hi=50.00 ALPHA-SATURATED
  target=+0.00 seed=0  lam=+0.6765  in_corridor=0%  acc=0.0651  DKY=4.83  a_hi=50.00 ALPHA-SATURATED
  target=+0.00 seed=1  lam=+0.5128  in_corridor=0%  acc=0.0696  DKY=6.95  a_hi=49.84 ALPHA-SATURATED
  target=+0.00 seed=2  lam=+0.4262  in_corridor=2%  acc=0.0658  DKY=0.00  a_hi=41.69
  target=+0.00 seed=3  lam=+0.5996  in_corridor=0%  acc=0.0626  DKY=1.78  a_hi=50.00 ALPHA-SATURATED
  target=+0.00 seed=4  lam=+1.1438  in_corridor=0%  acc=0.0612  DKY=33.62  a_hi=50.00 ALPHA-SATURATED
  t

In [18]:
g = SW.groupby("lam_target")
mean_acc, sd_acc = g.final_acc.mean(), g.final_acc.std()
mean_lam = g.lam_settled.mean()

fig, ax = plt.subplots(1, 3, figsize=(16, 4.4))
ax[0].errorbar(mean_acc.index, mean_acc.values, yerr=sd_acc.values, marker="o", capsize=4)
ax[0].axhline(CHANCE, color="r", ls="--", label="chance")
ax[0].axhline(FREE.final_acc.mean(), color="g", ls="--", label="unconstrained")
ax[0].axvline(0, color="k", ls=":", label=r"edge of chaos")
ax[0].set(xlabel=r"enforced $\lambda$ target", ylabel="copy accuracy",
          title="Does performance peak at the edge of chaos?")
ax[0].legend(); ax[0].grid(alpha=.3)

ax[1].plot(mean_lam.index, mean_lam.values, marker="o", label="achieved")
ax[1].plot(mean_lam.index, mean_lam.index, "k--", lw=1, label="requested")
ax[1].set(xlabel="requested target", ylabel=r"achieved $\hat\lambda$",
          title="Did dual ascent hold the constraint?")
ax[1].legend(); ax[1].grid(alpha=.3)

for tgt in CFG.lam_targets:
    d = SWH[SWH.lam_target == tgt].groupby("step")[["alpha_hi","alpha_lo"]].mean()
    ax[2].plot(d.index, d.alpha_hi + d.alpha_lo, label=f"{tgt:+.2f}")
ax[2].axhline(CFG.alpha_max, color="r", ls="--", label="cap")
ax[2].set(xlabel="step", ylabel=r"$\alpha_{hi}+\alpha_{lo}$", title="Multiplier trajectory", yscale="symlog")
ax[2].legend(fontsize=8); ax[2].grid(alpha=.3)
fig.tight_layout(); savefig(fig, "phase0c_sweep.png"); plt.close(fig)

saved /content/drive/My Drive/ChaoSSM/results/phase0c/phase0c_sweep.png


In [19]:
# Paired comparison of each target against the unconstrained reference
def vs_free(tgt):
    x = SW[SW.lam_target == tgt].sort_values("seed").final_acc.values
    y = FREE.sort_values("seed").final_acc.values[:len(x)]
    m = np.isfinite(x) & np.isfinite(y); x, y = x[m], y[m]
    if len(x) < 3 or np.allclose(x, y):
        return dict(target=tgt, n=len(x), p=np.nan,
                    median_diff=float(np.median(x-y)) if len(x) else np.nan)
    st, p = wilcoxon(x, y)
    return dict(target=tgt, n=len(x), p=float(p), median_diff=float(np.median(x-y)))

ST = pd.DataFrame([vs_free(t) for t in CFG.lam_targets])
print(ST.to_string(index=False)); savetable(ST, "phase0c_stats.csv")
print(f"\nmin achievable two-sided p at n={CFG.seeds}: {2/(2**CFG.seeds):.4f}")

best = mean_acc.idxmax()
print(f"\nbest-performing enforced target: lambda = {best:+.2f} "
      f"(acc {mean_acc.max():.4f} vs unconstrained {FREE.final_acc.mean():.4f})")

 target  n      p  median_diff
  -0.20  5 0.0625    -0.006299
   0.00  5 0.1250    -0.006543
   0.15  5 0.0625    -0.007959
   0.40  5 1.0000    -0.000537
   0.65  5 0.6250    -0.001172
saved /content/drive/My Drive/ChaoSSM/results/phase0c/phase0c_stats.csv

min achievable two-sided p at n=5: 0.0625

best-performing enforced target: lambda = +0.40 (acc 0.0712 vs unconstrained 0.0700)


## 6. Verdict

Read the constraint question and the performance question separately — they fail independently.

| Observation | Reading |
|---|---|
| `frac_in_corridor` high, α stays well below the cap | Dual ascent works. The corridor is enforceable and fixed-α was the only obstacle. Fold this into §4.4 as the training procedure. |
| α saturates at the cap and λ never reaches target | That target is unreachable for this architecture. Report the reachable range honestly rather than claiming the corridor. |
| Accuracy peaks near λ ≈ 0 | Direct support for §3.2. This is the result the proposal needs, and the sweep is the figure that carries the paper. |
| Accuracy flat across targets | Chaos is not doing work on this task. Either the task is too easy to discriminate, or the expressivity claim needs rethinking. Escalate to a harder long-range task before concluding. |
| Accuracy peaks at the unconstrained point | The LM objective already picks a good operating regime and the regularizer only costs. That is a negative result for §4.4 specifically, not for the chaotic map. |
| Learnability gate failed | Nothing in §5 is interpretable. Fix the task budget first. |

Whichever way it falls, the three corrections established in 0b — Option A's ceiling, the
single-step estimator bias, and the φ-vs-full-recurrence mismatch — hold regardless and belong
in the manuscript.

In [20]:
verdict = dict(
    smoke_test=SMOKE_TEST,
    chance=CHANCE,
    learnability_gate="PASS" if learned else "FAIL",
    unconstrained=dict(acc=float(FREE.final_acc.mean()),
                       lam=float(FREE.lam_settled.mean()),
                       DKY=float(FREE.DKY.mean())),
    sweep=SW.groupby("lam_target")[["lam_settled","frac_in_corridor","final_acc",
                                    "DKY","alpha_hi_final"]].mean().round(4).to_dict("index"),
    any_alpha_saturated=bool(SW.alpha_saturated.any()),
    best_target=float(best),
    stats=ST.to_dict("records"),
    config=dict(variant=CFG.variant, dual_lr=CFG.dual_lr, alpha_max=CFG.alpha_max,
                lyap_window=CFG.lyap_window, steps=CFG.steps, copy_len=CFG.copy_len),
)
savejson(verdict, "phase0c_verdict.json")
print(json.dumps(verdict, indent=2, default=str))

saved /content/drive/My Drive/ChaoSSM/results/phase0c/phase0c_verdict.json
{
  "smoke_test": false,
  "chance": 0.05555555555555555,
  "learnability_gate": "FAIL",
  "unconstrained": {
    "acc": 0.0700390625,
    "lam": 1.0313461738824845,
    "DKY": 32.762606706497586
  },
  "sweep": {
    "-0.2": {
      "lam_settled": 0.7519,
      "frac_in_corridor": 0.0,
      "final_acc": 0.0639,
      "DKY": 16.4608,
      "alpha_hi_final": 50.0
    },
    "0.0": {
      "lam_settled": 0.6718,
      "frac_in_corridor": 0.005,
      "final_acc": 0.0649,
      "DKY": 9.4364,
      "alpha_hi_final": 48.3046
    },
    "0.15": {
      "lam_settled": 0.726,
      "frac_in_corridor": 0.06,
      "final_acc": 0.0616,
      "DKY": 10.845,
      "alpha_hi_final": 35.4192
    },
    "0.4": {
      "lam_settled": 0.8092,
      "frac_in_corridor": 0.045,
      "final_acc": 0.0712,
      "DKY": 21.101,
      "alpha_hi_final": 29.1487
    },
    "0.65": {
      "lam_settled": 0.9041,
      "frac_in_corridor"